# TP3 – Contrôles qualité

Ryma Dinari – Retail Intelligence Platform

Étape 9 de l'UC-06 : je vérifie que les données Silver et Gold que j'ai produites sont correctes.
Chaque contrôle a une règle, un résultat et un statut (OK, ATTENTION ou KO). À la fin j'enregistre le tableau dans `preuves/controles_qualite.csv`.

In [1]:
import sys, csv
sys.path.insert(0, "..")

from config import *
from pyspark.sql import functions as F

spark = creer_session()

ventes = spark.read.parquet(str(SILVER / "sales"))
produits = spark.read.parquet(str(SILVER / "products"))
magasins = spark.read.parquet(str(SILVER / "stores"))
dates = spark.read.parquet(str(SILVER / "dates"))
gold_magasin = spark.read.parquet(str(GOLD / "ventes_magasin_mois"))
gold_categorie = spark.read.parquet(str(GOLD / "ventes_categorie_annee"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 15:09:08 WARN Utils: Your hostname, rymadinari-VivoBook-ASUSLaptop-M1702QA-M1702QA, resolves to a loopback address: 127.0.1.1; using 10.188.47.136 instead (on interface wlp1s0)
26/11/18 15:09:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 15:09:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/11/18 15:09:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
resultats = []

def controle(jeu, type_controle, regle, resultat, ok, attention=False):
    """Ajoute une ligne au rapport. attention=True : pas une erreur, mais un point connu à surveiller."""
    statut = "OK" if ok else ("ATTENTION" if attention else "KO")
    resultats.append((jeu, type_controle, regle, str(resultat), statut))
    print(f"[{statut:9}] {jeu} – {regle} : {resultat}")

## 1. Complétude

Les colonnes obligatoires ne doivent pas être vides.

In [3]:
obligatoires = {
    "silver/sales": (ventes, ["ss_sold_date_sk", "ss_item_sk", "ss_store_sk", "ss_ticket_number",
                              "ss_quantity", "ss_sales_price", "ss_net_paid"]),
    "silver/products": (produits, ["i_item_sk", "i_category"]),
    "silver/stores": (magasins, ["s_store_sk", "s_store_name"]),
    "silver/dates": (dates, ["d_date_sk", "d_date", "d_year", "d_moy"]),
}
for jeu, (df, colonnes) in obligatoires.items():
    for c in colonnes:
        nb = df.filter(F.col(c).isNull()).count()
        controle(jeu, "Complétude", f"{c} non nul", nb, nb == 0)

[OK       ] silver/sales – ss_sold_date_sk non nul : 0
[OK       ] silver/sales – ss_item_sk non nul : 0


[OK       ] silver/sales – ss_store_sk non nul : 0
[OK       ] silver/sales – ss_ticket_number non nul : 0


[OK       ] silver/sales – ss_quantity non nul : 0
[OK       ] silver/sales – ss_sales_price non nul : 0


[OK       ] silver/sales – ss_net_paid non nul : 0
[OK       ] silver/products – i_item_sk non nul : 0


[OK       ] silver/products – i_category non nul : 0
[OK       ] silver/stores – s_store_sk non nul : 0
[OK       ] silver/stores – s_store_name non nul : 0


[OK       ] silver/dates – d_date_sk non nul : 0
[OK       ] silver/dates – d_date non nul : 0
[OK       ] silver/dates – d_year non nul : 0


[OK       ] silver/dates – d_moy non nul : 0


In [4]:
# colonnes facultatives : je mesure le taux de vides, sans bloquer
for c in ["ss_promo_sk", "ss_addr_sk", "ss_cdemo_sk", "ss_hdemo_sk"]:
    nb = ventes.filter(F.col(c).isNull()).count()
    controle("silver/sales", "Complétude", f"{c} (facultatif) rempli",
             f"{nb} vides ({round(nb / ventes.count() * 100, 2)} %)", nb == 0, attention=True)

[ATTENTION] silver/sales – ss_promo_sk (facultatif) rempli : 158 vides (0.02 %)
[ATTENTION] silver/sales – ss_addr_sk (facultatif) rempli : 448 vides (0.07 %)


[ATTENTION] silver/sales – ss_cdemo_sk (facultatif) rempli : 238 vides (0.04 %)
[ATTENTION] silver/sales – ss_hdemo_sk (facultatif) rempli : 295 vides (0.04 %)


## 2. Unicité et clés candidates

In [5]:
def nb_doublons(df, cle):
    return df.count() - df.select(*cle).distinct().count()

controle("silver/products", "Unicité", "i_item_sk unique", nb_doublons(produits, ["i_item_sk"]),
         nb_doublons(produits, ["i_item_sk"]) == 0)
controle("silver/stores", "Unicité", "s_store_sk unique", nb_doublons(magasins, ["s_store_sk"]),
         nb_doublons(magasins, ["s_store_sk"]) == 0)
controle("silver/dates", "Unicité", "d_date_sk unique", nb_doublons(dates, ["d_date_sk"]),
         nb_doublons(dates, ["d_date_sk"]) == 0)

colonnes_metier = [c for c in ventes.columns if c not in ("source_fichier", "date_traitement")]
nb = nb_doublons(ventes, colonnes_metier)
controle("silver/sales", "Unicité", "aucune ligne totalement identique", nb, nb == 0)

nb = nb_doublons(ventes, ["ss_ticket_number", "ss_item_sk"])
controle("silver/sales", "Clé candidate", "(ticket, produit) unique", nb, nb == 0, attention=True)

[OK       ] silver/products – i_item_sk unique : 0


[OK       ] silver/stores – s_store_sk unique : 0


[OK       ] silver/dates – d_date_sk unique : 0


[OK       ] silver/sales – aucune ligne totalement identique : 0


[ATTENTION] silver/sales – (ticket, produit) unique : 4136


Le couple (ticket, produit) n'est pas unique, mais c'est connu depuis le notebook 01 : ce sont des vraies ventes (même produit passé deux fois sur un ticket). Je le mets en ATTENTION et pas en KO.

## 3. Validité des types

Les types en Silver doivent être ceux définis dans `config.py`.

In [6]:
types_spark = {"bigint": "bigint", "int": "int", "string": "string", "date": "date"}

for jeu, df, table in [("silver/sales", ventes, "store_sales"), ("silver/products", produits, "item"),
                       ("silver/stores", magasins, "store"), ("silver/dates", dates, "date_dim")]:
    reels = dict(df.dtypes)
    erreurs = [c for c, t in COLONNES[table] if c in reels and reels[c] != types_spark.get(t, t)]
    controle(jeu, "Validité", "types conformes à config.py", f"{len(erreurs)} colonne(s) différente(s)",
             len(erreurs) == 0)

[OK       ] silver/sales – types conformes à config.py : 0 colonne(s) différente(s)
[OK       ] silver/products – types conformes à config.py : 0 colonne(s) différente(s)
[OK       ] silver/stores – types conformes à config.py : 0 colonne(s) différente(s)
[OK       ] silver/dates – types conformes à config.py : 0 colonne(s) différente(s)


## 4. Cohérence des valeurs

In [7]:
regles = [
    ("quantité entre 1 et 100", "ss_quantity < 1 or ss_quantity > 100"),
    ("montant payé >= 0", "ss_net_paid < 0"),
    ("prix de vente <= prix catalogue", "ss_sales_price > ss_list_price"),
    ("montant brut = quantité x prix", "abs(ss_ext_sales_price - ss_quantity * ss_sales_price) > 0.01"),
    ("net payé = montant brut - coupon", "abs(ss_net_paid - (ss_ext_sales_price - ss_coupon_amt)) > 0.01"),
]
for regle, condition_erreur in regles:
    nb = ventes.filter(condition_erreur).count()
    controle("silver/sales", "Cohérence", regle, f"{nb} ligne(s) en erreur", nb == 0)

# les dates de vente doivent être dans la période du kit
periode = ventes.join(dates, ventes.ss_sold_date_sk == dates.d_date_sk) \
                .select(F.min("d_date").alias("debut"), F.max("d_date").alias("fin")).first()
ok = str(periode.debut) >= "2001-01-01" and str(periode.fin) <= "2005-12-31"
controle("silver/sales", "Cohérence", "ventes entre 2001 et 2005", f"{periode.debut} -> {periode.fin}", ok)

nb = ventes.filter("ss_net_profit < 0").count()
controle("silver/sales", "Cohérence", "marge positive",
         f"{nb} marges négatives ({round(nb / ventes.count() * 100)} %)", nb == 0, attention=True)

[OK       ] silver/sales – quantité entre 1 et 100 : 0 ligne(s) en erreur
[OK       ] silver/sales – montant payé >= 0 : 0 ligne(s) en erreur


[OK       ] silver/sales – prix de vente <= prix catalogue : 0 ligne(s) en erreur


[OK       ] silver/sales – montant brut = quantité x prix : 0 ligne(s) en erreur
[OK       ] silver/sales – net payé = montant brut - coupon : 0 ligne(s) en erreur


[OK       ] silver/sales – ventes entre 2001 et 2005 : 2001-01-01 -> 2005-12-13


[ATTENTION] silver/sales – marge positive : 495971 marges négatives (74 %)


## 5. Volumes

On ne doit perdre aucune ligne entre le fichier brut et le Silver.

In [8]:
for jeu, df, table in [("silver/sales", ventes, "store_sales"), ("silver/products", produits, "item"),
                       ("silver/stores", magasins, "store"), ("silver/dates", dates, "date_dim")]:
    brut = lire_brut(spark, table).count()
    silver = df.count()
    controle(jeu, "Volume", "lignes Silver = lignes brutes", f"{silver} / {brut}", silver == brut)

[OK       ] silver/sales – lignes Silver = lignes brutes : 667579 / 667579
[OK       ] silver/products – lignes Silver = lignes brutes : 17820 / 17820


[OK       ] silver/stores – lignes Silver = lignes brutes : 12 / 12
[OK       ] silver/dates – lignes Silver = lignes brutes : 109573 / 109573


## 6. Intégrité référentielle

In [9]:
liens = [
    ("ss_item_sk", produits, "i_item_sk", "products"),
    ("ss_store_sk", magasins, "s_store_sk", "stores"),
    ("ss_sold_date_sk", dates, "d_date_sk", "dates"),
]
for cle, ref, cle_ref, nom in liens:
    nb = ventes.join(ref, ventes[cle] == ref[cle_ref], "left_anti").count()
    controle("silver/sales", "Intégrité", f"{cle} existe dans {nom}", f"{nb} orphelin(s)", nb == 0)

[OK       ] silver/sales – ss_item_sk existe dans products : 0 orphelin(s)
[OK       ] silver/sales – ss_store_sk existe dans stores : 0 orphelin(s)


[OK       ] silver/sales – ss_sold_date_sk existe dans dates : 0 orphelin(s)


## 7. Contrôles sur le Gold

In [10]:
ca_silver = ventes.agg(F.sum("ss_net_paid")).first()[0]
ca_gold1 = gold_magasin.agg(F.sum("ca")).first()[0]
ca_gold2 = gold_categorie.agg(F.sum("ca")).first()[0]
controle("gold/ventes_magasin_mois", "Cohérence", "CA total = CA Silver", f"{ca_gold1} / {ca_silver}", ca_gold1 == ca_silver)
controle("gold/ventes_categorie_annee", "Cohérence", "CA total = CA Silver", f"{ca_gold2} / {ca_silver}", ca_gold2 == ca_silver)

nb = gold_magasin.count()
controle("gold/ventes_magasin_mois", "Volume", "12 magasins x 60 mois = 720 lignes", nb, nb == 720)

nb = nb_doublons(gold_magasin, ["s_store_sk", "annee", "mois"])
controle("gold/ventes_magasin_mois", "Unicité", "(magasin, année, mois) unique", nb, nb == 0)

nb = nb_doublons(gold_categorie, ["categorie", "annee"])
controle("gold/ventes_categorie_annee", "Unicité", "(catégorie, année) unique", nb, nb == 0)

parts = gold_categorie.groupBy("annee").agg(F.sum("part_ca_pct").alias("total"))
nb = parts.filter("abs(total - 100) > 0.1").count()
controle("gold/ventes_categorie_annee", "Cohérence", "parts de CA = 100 % par année", f"{nb} année(s) en erreur", nb == 0)

[OK       ] gold/ventes_magasin_mois – CA total = CA Silver : 1150529586.37 / 1150529586.37
[OK       ] gold/ventes_categorie_annee – CA total = CA Silver : 1150529586.37 / 1150529586.37
[OK       ] gold/ventes_magasin_mois – 12 magasins x 60 mois = 720 lignes : 720


[OK       ] gold/ventes_magasin_mois – (magasin, année, mois) unique : 0


[OK       ] gold/ventes_categorie_annee – (catégorie, année) unique : 0


[OK       ] gold/ventes_categorie_annee – parts de CA = 100 % par année : 0 année(s) en erreur


## 8. Rapport

In [11]:
rapport = spark.createDataFrame(resultats, "jeu string, controle string, regle string, resultat string, statut string")
rapport.show(100, truncate=False)
rapport.groupBy("statut").count().show()

with open(PREUVES / "controles_qualite.csv", "w", newline="", encoding="utf-8") as f:
    ecrivain = csv.writer(f, delimiter=";")
    ecrivain.writerow(["jeu", "controle", "regle", "resultat", "statut"])
    ecrivain.writerows(resultats)
print("rapport enregistré dans", PREUVES / "controles_qualite.csv")

+---------------------------+-------------+----------------------------------+------------------------------+---------+
|jeu                        |controle     |regle                             |resultat                      |statut   |
+---------------------------+-------------+----------------------------------+------------------------------+---------+
|silver/sales               |Complétude   |ss_sold_date_sk non nul           |0                             |OK       |
|silver/sales               |Complétude   |ss_item_sk non nul                |0                             |OK       |
|silver/sales               |Complétude   |ss_store_sk non nul               |0                             |OK       |
|silver/sales               |Complétude   |ss_ticket_number non nul          |0                             |OK       |
|silver/sales               |Complétude   |ss_quantity non nul               |0                             |OK       |
|silver/sales               |Complétude 

+---------+-----+
|   statut|count|
+---------+-----+
|       OK|   42|
|ATTENTION|    6|
+---------+-----+

rapport enregistré dans /home/rymadinari/TP3/preuves/controles_qualite.csv


## Bilan

**Résultat : 42 contrôles OK, 6 ATTENTION, 0 KO.**

- **Règles respectées** : colonnes obligatoires remplies, clés des tables de référence uniques, types conformes, montants cohérents entre eux, aucune ligne perdue entre le brut et le Silver, aucune vente orpheline, totaux du Gold égaux au Silver.
- **Anomalies restantes (ATTENTION)** :
  - quelques clés facultatives vides dans les ventes (moins de 0,1 %), elles ne servent pas à mes indicateurs ;
  - 4 136 doublons (ticket, produit), gardés car ce sont de vraies ventes ;
  - 74 % de marges négatives, dues au générateur TPCx-BB.
- **Données mises à l'écart** : aucune (le dossier `data/quarantine` est vide).
- **Limites des contrôles** : je vérifie la cohérence interne des données, pas leur vérité (ce sont des données inventées). Je n'ai contrôlé que les 4 sources que j'utilise, et seulement le lot principal (`data/`, pas `data_refresh/`).
- **Corrections nécessaires** : aucune pour le moment. Si on ajoute le lot `data_refresh`, il faudra relancer ces contrôles, notamment les doublons avec les ventes déjà présentes.